# Stage 5 — Persona Adaptation

**Target Generation:** Groq free API (Llama-3.1-70B) — produces genuinely distinct outputs per persona  
**Fine-tuning:** flan-t5-base on the generated pairs  


In [ ]:
!pip install -q \
    transformers==4.40.2 \
    huggingface_hub==0.23.4 \
    accelerate==0.29.3 \
    groq \
    textstat \
    sentencepiece

print("\n✅ Done. Restart kernel, then run from the next cell.")

In [ ]:
# Verify versions
import transformers, huggingface_hub
print(f"transformers   : {transformers.__version__}")
print(f"huggingface_hub: {huggingface_hub.__version__}")
assert transformers.__version__.startswith("4.40"),  "Re-run install cell + restart kernel"
assert huggingface_hub.__version__.startswith("0.23"), "Re-run install cell + restart kernel"
print("✅ Versions OK")

In [ ]:
import json, re, random, gc, time
from pathlib import Path
from collections import Counter
import torch

INPUT_PATH   = "/kaggle/input/datasets/harshitgoyal41/stage3output/simplified_test_clauses_stage3.json"
TARGET_PATH  = "/kaggle/working/persona_targets.json"
TRAIN_PATH   = "/kaggle/working/train_pairs.json"
OUTPUT_MODEL = "/kaggle/working/persona_adapter"
PERSONAS     = ["layperson", "student", "professional"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

with open(INPUT_PATH) as f:
    raw_data = json.load(f)

def is_valid(item):
    if item.get("immutable", False): return False
    text = (item.get("simplified") or "").strip()
    if len(text.split()) < 15: return False
    if any(k in text.upper().split()[:3] for k in ["EXHIBIT", "SCHEDULE"]): return False
    return True

def clean_text(text):
    text = re.sub(r"EXPLAINED.*?\n", "", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"(agreement)\s+\1", r"\1", text, flags=re.IGNORECASE)
    return text.strip()

filtered = [d for d in raw_data if is_valid(d)]
for item in filtered:
    item["simplified"] = clean_text(item["simplified"])

print(f"Raw: {len(raw_data)} | Filtered: {len(filtered)}")
print(f"Obligations: {Counter(d['obligation'] for d in filtered)}")

## Step 2 — Generate Persona Targets via Groq (Llama-3.1-70B)

Llama-3.1-70B is strong enough to produce genuinely different outputs for each persona level — unlike flan-t5-large which produced near-identical outputs.

In [ ]:
from groq import Groq
from kaggle_secrets import UserSecretsClient

GROQ_API_KEY = UserSecretsClient().get_secret("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)

SYSTEM_PROMPT = """You are a legal language expert specializing in document accessibility.
Given a legal clause, rewrite it in 3 clearly distinct styles.
Return ONLY a valid JSON object. No markdown, no explanation, no preamble.

JSON format:
{
  "layperson": "...",
  "student": "...",
  "professional": "..."
}

Style guidelines (these must be CLEARLY different from each other):
- layperson: Imagine explaining to a 12-year-old. Use everyday words only. No legal terms at all. Short simple sentences. 1-2 sentences max.
- student: Written for a 2nd-year law student. Use legal terms but define the important ones in brackets. 2-3 sentences.
- professional: Written for a senior lawyer reviewing a contract. Dense, precise, uses standard legal terminology with no hand-holding. 1-2 sentences."""

def generate_persona_targets(clause_text: str, obligation: str, retries: int = 3) -> dict | None:
    user_msg = f"Obligation type: {obligation}\n\nClause:\n{clause_text[:700]}"
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.1-70b-versatile",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg}
                ],
                temperature=0.7,     # some variation helps persona diversity
                max_tokens=400,
                response_format={"type": "json_object"}  # forces valid JSON output
            )
            raw    = response.choices[0].message.content.strip()
            parsed = json.loads(raw)

            # Validate all 3 keys present and non-trivially different
            if not all(k in parsed for k in PERSONAS):
                continue
            if not all(isinstance(parsed[k], str) and len(parsed[k].split()) >= 5 for k in PERSONAS):
                continue

            # Check they are actually different (not copy-pasted)
            vals = [parsed[k].lower().strip() for k in PERSONAS]
            if vals[0] == vals[1] or vals[1] == vals[2]:
                # retry if identical
                continue

            return parsed

        except Exception as e:
            wait = 2 ** attempt   # exponential backoff: 1s, 2s, 4s
            print(f"  Attempt {attempt+1} failed: {e} — retrying in {wait}s")
            time.sleep(wait)

    return None

# ── Smoke test ────────────────────────────────────────────────────────────────
test = generate_persona_targets(
    "The Company must pay $10,000 within 30 days of invoice receipt.",
    "MANDATORY"
)
print("Smoke test — are outputs clearly different?")
for p in PERSONAS:
    print(f"\n  [{p.upper()}]")
    print(f"  {test[p]}")

In [ ]:
from tqdm.auto import tqdm
import json, time
from pathlib import Path

# ── Resume from checkpoint ────────────────────────────────────────────────────
if Path(TARGET_PATH).exists():
    with open(TARGET_PATH) as f:
        targets = json.load(f)
    done_ids = {(t["doc_id"], t["clause_idx"]) for t in targets}
    print(f"Resuming — {len(targets)} already done")
else:
    targets, done_ids = [], set()

SAVE_EVERY = 25
fallbacks  = 0


# ── Generator (FIXED) ─────────────────────────────────────────────────────────
def generate_persona_targets(clause_text, obligation, retries=3):
    user_msg = f"Obligation type: {obligation}\n\nClause:\n{clause_text[:700]}"

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="openai/gpt-oss-120b",   # ✅ your best working model
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg}
                ],
                temperature=0.5,
                max_tokens=400,
                response_format={"type": "json_object"}   # ✅ KEY FIX
            )

            raw = response.choices[0].message.content.strip()
            parsed = json.loads(raw)

            # ✅ Check keys exist
            if not all(k in parsed for k in PERSONAS):
                continue

            # ✅ Relaxed validation (important)
            if not all(isinstance(parsed[k], str) and len(parsed[k].strip()) > 0 for k in PERSONAS):
                continue

            # ✅ Strong diversity check
            vals = [parsed[k].lower().strip() for k in PERSONAS]
            if len(set(vals)) < 3:
                continue

            return parsed

        except Exception as e:
            wait = 2 ** attempt
            print(f"  Attempt {attempt+1} failed: {e} — retrying in {wait}s")
            time.sleep(wait)

    return None


# ── Main loop ────────────────────────────────────────────────────────────────
for i, item in enumerate(tqdm(filtered, desc="Generating persona targets")):
    uid = (item["doc_id"], item["clause_idx"])

    if uid in done_ids:
        continue

    result = generate_persona_targets(
        item["simplified"],
        item["obligation"]
    )

    if result:
        targets.append({
            "doc_id":       item["doc_id"],
            "clause_idx":   item["clause_idx"],
            "simplified":   item["simplified"],
            "obligation":   item["obligation"],
            "layperson":    result["layperson"],
            "student":      result["student"],
            "professional": result["professional"]
        })
    else:
        fallbacks += 1
        tqdm.write(f"  Skipped clause {item['clause_idx']} after retries")

    # ⚡ light throttle (Groq free tier safety)
    time.sleep(0.5)

    if (i + 1) % SAVE_EVERY == 0:
        with open(TARGET_PATH, "w") as f:
            json.dump(targets, f, indent=2)
        tqdm.write(f"  Checkpoint: {len(targets)} done, {fallbacks} skipped")


# ── Final save ────────────────────────────────────────────────────────────────
with open(TARGET_PATH, "w") as f:
    json.dump(targets, f, indent=2)

print(f"\nDone. Targets: {len(targets)} | Skipped: {fallbacks}")

# ── Safe sample print ─────────────────────────────────────────────────────────
if len(targets) > 5:
    print("\nSample output (clause 5):")
    for p in PERSONAS:
        print(f"  [{p.upper():12s}]: {targets[5][p]}")
else:
    print("\nNot enough samples to display example.")

## Step 3 — Build Training Pairs

In [ ]:
with open(TARGET_PATH) as f:
    targets = json.load(f)

train_pairs = []
for item in targets:
    for persona in PERSONAS:
        target_text = (item.get(persona) or "").strip()
        if not target_text or len(target_text.split()) < 5:
            continue
        train_pairs.append({
            "input":      f"adapt for {persona}: {item['simplified']}",
            "target":     target_text,
            "persona":    persona,
            "obligation": item["obligation"]
        })

with open(TRAIN_PATH, "w") as f:
    json.dump(train_pairs, f, indent=2)

print(f"Total training pairs: {len(train_pairs)}")
print(f"Per persona: {Counter(p['persona'] for p in train_pairs)}")
print(f"\nSample input : {train_pairs[0]['input'][:200]}")
print(f"Sample target: {train_pairs[0]['target'][:200]}")

## Step 4 — Load flan-t5-base + Dataset

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
from torch.utils.data import Dataset, DataLoader

FT_MODEL      = "google/flan-t5-base"
MAX_IN_LEN    = 256
MAX_TGT_LEN   = 128
BATCH_SIZE    = 8
LEARNING_RATE = 5e-5
EPOCHS        = 3

tokenizer = T5Tokenizer.from_pretrained(FT_MODEL)
model     = T5ForConditionalGeneration.from_pretrained(FT_MODEL).to(device)
print(f"flan-t5-base loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

class PersonaDataset(Dataset):
    def __init__(self, pairs, tok, max_in, max_tgt):
        self.pairs, self.tok = pairs, tok
        self.max_in, self.max_tgt = max_in, max_tgt

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        p   = self.pairs[idx]
        enc = self.tok(p["input"],  max_length=self.max_in,  padding="max_length", truncation=True, return_tensors="pt")
        tgt = self.tok(p["target"], max_length=self.max_tgt, padding="max_length", truncation=True, return_tensors="pt")
        lbl = tgt["input_ids"].squeeze().clone()
        lbl[lbl == self.tok.pad_token_id] = -100
        return {"input_ids":      enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "labels":         lbl}

random.seed(42)
random.shuffle(train_pairs)
split = int(0.9 * len(train_pairs))

train_loader = DataLoader(PersonaDataset(train_pairs[:split], tokenizer, MAX_IN_LEN, MAX_TGT_LEN),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(PersonaDataset(train_pairs[split:],  tokenizer, MAX_IN_LEN, MAX_TGT_LEN),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {split} pairs ({len(train_loader)} batches)")
print(f"Val  : {len(train_pairs)-split} pairs ({len(val_loader)} batches)")

## Step 5 — Train

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

optimizer   = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(optimizer, total_steps // 10, total_steps)
best_val    = float("inf")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]")
    for batch in loop:
        optimizer.zero_grad()
        loss = model(input_ids=batch["input_ids"].to(device),
                     attention_mask=batch["attention_mask"].to(device),
                     labels=batch["labels"].to(device)).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()
        loop.set_postfix(loss=f"{loss.item():.4f}")

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [val]", leave=False):
            val_loss += model(input_ids=batch["input_ids"].to(device),
                              attention_mask=batch["attention_mask"].to(device),
                              labels=batch["labels"].to(device)).loss.item()

    avg_t = train_loss / len(train_loader)
    avg_v = val_loss   / len(val_loader)
    print(f"\nEpoch {epoch+1} → Train: {avg_t:.4f} | Val: {avg_v:.4f}")
    if avg_v < best_val:
        best_val = avg_v
        model.save_pretrained(OUTPUT_MODEL)
        tokenizer.save_pretrained(OUTPUT_MODEL)
        print(f"  ✅ Best saved (val={best_val:.4f})")

print("\nTraining complete.")

## Step 6 — Inference + Readability Evaluation

In [ ]:
import textstat
from transformers import T5Tokenizer, T5ForConditionalGeneration

best_model = T5ForConditionalGeneration.from_pretrained(OUTPUT_MODEL).to(device)
best_tok   = T5Tokenizer.from_pretrained(OUTPUT_MODEL)
best_model.eval()

def generate(text: str, persona: str, max_new_tokens: int = 150) -> str:
    inputs = best_tok(
        f"adapt for {persona}: {text}",
        return_tensors="pt", truncation=True, max_length=256
    ).to(device)
    with torch.no_grad():
        out = best_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            repetition_penalty=2.5,
            no_repeat_ngram_size=3,
            early_stopping=True
        )
    return best_tok.decode(out[0], skip_special_tokens=True)

test_clauses = [
    ("The Company must pay $10,000 within 30 days of invoice receipt.",       "MANDATORY"),
    ("Either party may terminate this Agreement upon 60 days written notice.", "CONDITIONAL"),
    ("ENVISION owns all patents for the product, including future patents.",   "DECLARATIVE"),
]

for clause, ob in test_clauses:
    print(f"\n{'='*65}")
    print(f"ORIGINAL [{ob}]: {clause}")
    print(f"{'='*65}")
    for persona in PERSONAS:
        out = generate(clause, persona)
        fk  = textstat.flesch_kincaid_grade(out)
        fre = textstat.flesch_reading_ease(out)
        print(f"  [{persona.upper():12s}]: {out}")
        print(f"  {'':14s}  FK={fk:.1f}  FRE={fre:.1f}")

print("\nExpected readability ranges:")
print("  layperson    → FK < 6,   FRE > 65")
print("  student      → FK 6–10,  FRE 45–65")
print("  professional → FK > 10,  FRE < 45")

## Step 7 — Full Inference on All Clauses

In [ ]:
results = []
for item in tqdm(filtered, desc="Inference"):
    entry = {
        "doc_id":     item["doc_id"],
        "clause_idx": item["clause_idx"],
        "original":   item.get("original", ""),
        "simplified": item["simplified"],
        "obligation": item["obligation"]
    }
    for persona in PERSONAS:
        entry[persona] = generate(item["simplified"], persona)
    results.append(entry)

with open("/kaggle/working/stage5_output.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved {len(results)} results.")
print(json.dumps(results[10], indent=2))

## Step 8 — Push to HuggingFace Hub (Optional)

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

login(UserSecretsClient().get_secret("HF_TOKEN_PUSH"))

HF_REPO = "harshitgoyal41/legal-persona-adapter-flan-t5-base"
best_model.push_to_hub(HF_REPO)
best_tok.push_to_hub(HF_REPO)
print(f"Pushed → https://huggingface.co/{HF_REPO}")